# 01 — Data Exploration
**DeepLense GSoC 2026 — Pallab Mondal**

This notebook explores the simulated gravitational lensing datasets (Model I, II, III).  
Goals:
- Visualise sample images from each dark-matter class
- Check class balance and image statistics
- Apply and visualise the physics-informed preprocessing map
- Visualise lensing inversion (source reconstruction)

In [ ]:
# ── Standard imports ─────────────────────────────────────────────────────────
import sys, os
from pathlib import Path

# Add my_work to path
MY_WORK = Path(os.getcwd()).parent      # notebooks/../  = my_work/
sys.path.insert(0, str(MY_WORK))

import numpy as np
import matplotlib.pyplot as plt
import torch

from config import (
    MODEL_I_TRAIN, MODEL_I_TEST, CLASS_NAMES, IMAGE_SIZE
)
from utils.data_loader import DeepLenseDataset, get_val_transform
from utils.physics import LensingInversionLayer, PhysicsPreprocessing

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}  |  Device: {device}')

In [ ]:
# ── Load Model I training set ─────────────────────────────────────────────────
train_ds = DeepLenseDataset(
    root=MODEL_I_TRAIN,
    class_names=CLASS_NAMES[:3],   # Model I has 3 classes
    transform=get_val_transform(IMAGE_SIZE),
)
print(f'Total samples: {len(train_ds)}')

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
from collections import Counter

label_counts = Counter([label for _, label in train_ds.samples])
class_labels = [CLASS_NAMES[l] for l in sorted(label_counts)]
counts = [label_counts[l] for l in sorted(label_counts)]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(class_labels, counts, color=['#4C9BE8', '#E86C4C', '#50C878'])
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{cnt:,}', ha='center', fontsize=10)
ax.set(title='Model I — Class Distribution', ylabel='Number of images')
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualise sample images ───────────────────────────────────────────────────
import random
random.seed(42)

n_per_class = 4
active_classes = CLASS_NAMES[:3]

fig, axes = plt.subplots(len(active_classes), n_per_class, figsize=(12, 9))
fig.suptitle('Model I — Sample Images per Class', fontsize=14, y=1.01)

for row, cls in enumerate(active_classes):
    cls_idx = CLASS_NAMES.index(cls)
    indices = [i for i, (_, l) in enumerate(train_ds.samples) if l == cls_idx]
    chosen  = random.sample(indices, min(n_per_class, len(indices)))
    
    for col, idx in enumerate(chosen):
        img, _ = train_ds[idx]
        ax = axes[row][col]
        ax.imshow(img.squeeze().numpy(), cmap='inferno')
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(cls.replace('_', ' ').title(), fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ── Physics preprocessing visualisation ──────────────────────────────────────
preproc = PhysicsPreprocessing().to(device)
lensing = LensingInversionLayer().to(device)

# Pick one sample per class
fig, axes = plt.subplots(len(active_classes), 3, figsize=(10, 9))
col_titles = ['Original', 'Physics Preprocessing', 'Source Reconstruction']
for ax, t in zip(axes[0], col_titles):
    ax.set_title(t, fontsize=10)

theta_E_demo = torch.tensor([[0.8]]).to(device)  # typical Einstein radius

for row, cls in enumerate(active_classes):
    cls_idx = CLASS_NAMES.index(cls)
    idx = next(i for i, (_, l) in enumerate(train_ds.samples) if l == cls_idx)
    img, _ = train_ds[idx]
    img_t = img.unsqueeze(0).to(device)  # (1, 1, 64, 64)

    with torch.no_grad():
        preprocessed = preproc(img_t)
        source       = lensing(img_t, theta_E_demo)

    def show(ax, tensor, cmap='inferno'):
        ax.imshow(tensor.squeeze().cpu().numpy(), cmap=cmap)
        ax.axis('off')

    show(axes[row][0], img_t)
    show(axes[row][1], preprocessed, cmap='RdBu')
    show(axes[row][2], source)
    axes[row][0].set_ylabel(cls.replace('_', ' ').title(), fontsize=9)

plt.suptitle('Physics-Informed Preprocessing  |  θ_E = 0.8 arcsec', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Pixel intensity statistics ────────────────────────────────────────────────
from torch.utils.data import DataLoader

loader = DataLoader(train_ds, batch_size=256, num_workers=0)
all_means, all_stds, all_maxs = [], [], []

for imgs, _ in loader:
    flat = imgs.view(imgs.size(0), -1)
    all_means.extend(flat.mean(dim=1).tolist())
    all_stds .extend(flat.std(dim=1).tolist())
    all_maxs .extend(flat.max(dim=1).values.tolist())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, label in zip(
    axes,
    [all_means, all_stds, all_maxs],
    ['Per-image Mean', 'Per-image Std', 'Per-image Max'],
):
    ax.hist(data, bins=60, edgecolor='none', alpha=0.8, color='#4C9BE8')
    ax.set(title=label, xlabel='Pixel value', ylabel='Count')

plt.suptitle('Model I — Pixel Statistics', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Overall mean: {np.mean(all_means):.4f}')
print(f'Overall std:  {np.mean(all_stds):.4f}')